In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

In [3]:
df = pd.read_csv('final_glof_prediction.csv')

df.head()

,lake_id,lake_area_km2,expansion_rate_km2_yr,elevation_m,area_uncertainty_km2,peak_discharge_m3s
0,GLO_80.85718_29.91975,0.004988,0.0000,4782.9,0.003,121.69
1,GLO_80.88324_30.0535,0.034663,0.0005,4371.9,0.008,494.55
2,GLO_80.88712_30.04913,0.021446,0.0000,4418.7,0.006,345.66
3,GLO_80.90401_29.97394,0.029676,-0.0001,4202.9,0.007,431.89
4,GLO_80.95473_29.86154,0.030756,-0.0011,4725.0,0.008,472.37


In [4]:

FEATURES = ['lake_area_km2', 'expansion_rate_km2_yr', 'elevation_m', 'area_uncertainty_km2']
TARGET   = 'peak_discharge_m3s'

X = df[FEATURES]
y = df[TARGET]

print('Features shape:', X.shape)
print('Target shape  :', y.shape)

Features shape: (1768, 4)
Target shape  : (1768,)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Training samples : {len(X_train)}')
print(f'Test samples     : {len(X_test)}')

Training samples : 1414
Test samples     : 354


In [6]:
model = LinearRegression()
model.fit(X_train, y_train)

print('Intercept :', round(model.intercept_, 4))
print('\nCoefficients:')
for feat, coef in zip(FEATURES, model.coef_):
    print(f'  {feat:<30s}: {coef:.4f}')

Intercept : -414.379

Coefficients:
  lake_area_km2                 : 10120.0736
  expansion_rate_km2_yr         : 59814.0401
  elevation_m                   : 0.0984
  area_uncertainty_km2          : 9152.3812


In [7]:

y_pred = model.predict(X_test)

mse  = mean_squared_error(y_test, y_pred)
r2   = r2_score(y_test, y_pred)

print('Test-set Metrics')
print(f'  R²   : {r2:.4f}')
print(f'  MSE  : {mse:.4f}')

Test-set Metrics
  R²   : 0.9854
  MSE  : 62013.0762


In [8]:
results = pd.DataFrame({
    'Actual (m³/s)'   : y_test.values[:10],
    'Predicted (m³/s)': y_pred[:10].round(2),
    'Error'           : (y_test.values[:10] - y_pred[:10]).round(2)
})
print(results.to_string(index=False))

 Actual (m³/s)  Predicted (m³/s)   Error
         48.00             68.74  -20.74
        122.71            187.59  -64.88
        305.85            256.45   49.40
        151.33             34.47  116.86
        357.69            268.69   89.00
       5429.23           5479.72  -50.49
        205.05            178.55   26.50
         42.33             91.81  -49.48
       1727.50           1928.29 -200.79
         99.98             58.94   41.04


In [9]:
rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

print('Feature importances:')
for f, imp in zip(FEATURES, rf.feature_importances_):
    print(f'  {f:<30s}: {imp:.4f}')

Feature importances:
  lake_area_km2                 : 0.5998
  expansion_rate_km2_yr         : 0.0374
  elevation_m                   : 0.0069
  area_uncertainty_km2          : 0.3559


In [10]:
y_pred = rf.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print('Test-set Metrics')
print(f'  R²   : {r2:.4f}')
print(f'  MSE  : {mse:.4f}')

Test-set Metrics
  R²   : 0.9176
  MSE  : 349436.3322


In [11]:
results = pd.DataFrame({
    'Actual (m³/s)': y_test.values[:10],
    'Predicted (m³/s)': y_pred[:10].round(2),
    'Error': (y_test.values[:10] - y_pred[:10]).round(2)
})
print(results.to_string(index=False))

 Actual (m³/s)  Predicted (m³/s)  Error
         48.00             48.49  -0.49
        122.71            122.95  -0.24
        305.85            307.71  -1.86
        151.33            168.18 -16.85
        357.69            361.77  -4.08
       5429.23           5101.83 327.40
        205.05            205.37  -0.32
         42.33             42.40  -0.07
       1727.50           1811.25 -83.75
         99.98            101.34  -1.36


In [13]:

from sklearn.model_selection import KFold, cross_validate

cv = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    )
}

scoring = {
    "R2": "r2",
}

results = {}

for name, model in models.items():

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        return_train_score=True,
        n_jobs=-1
    )

    results[name] = {
        "Train R2": scores["train_R2"].mean(),
        "Validation R2": scores["test_R2"].mean(),
        "R2 Std": scores["test_R2"].std()
    }

cv_results = pd.DataFrame(results).T

print("5-Fold Cross-Validation Results")
print(cv_results.round(4))


5-Fold Cross-Validation Results
                   Train R2  Validation R2  R2 Std
Linear Regression    0.9867         0.9687  0.0179
Random Forest        0.9744         0.9218  0.0795
